# Colab HF/BnB LoRA Eval

Runs the same `sc_terse` + `high_div_8` evaluation path as the repo CLI, with the adapter loaded from Google Drive. Start with the 5-item speed test, then choose `MAX_TOKENS` for the full base and adapter runs.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Repo Setup

Set `REPO_URL` if you want the notebook to clone the project into Colab. If you already copied the repo to `/content/151B_SP26_Competition`, leave it blank.

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_DIR = Path('/content/151B_SP26_Competition')
REPO_URL = ''  # Example: 'https://github.com/YOUR_USER/151B_SP26_Competition.git'

if not REPO_DIR.exists():
    if not REPO_URL:
        raise RuntimeError('Set REPO_URL above, or upload/copy the repo to /content/151B_SP26_Competition')
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print('cwd =', Path.cwd())

In [ ]:
%pip install -q "hydra-core>=1.3" peft bitsandbytes datasets accelerate transformers tqdm sympy pyyaml wandb

In [ ]:
import importlib

for name in ['hydra', 'omegaconf', 'torch', 'transformers', 'peft', 'bitsandbytes', 'datasets', 'tqdm', 'wandb']:
    module = importlib.import_module(name)
    print(name, getattr(module, '__version__', 'OK'))

## Adapter and Run Config

Update `DRIVE_ADAPTER_PATH` to the `final_adapter` folder in Google Drive. It should contain `adapter_config.json` and `adapter_model.safetensors`.

In [ ]:
from pathlib import Path

DRIVE_ADAPTER_PATH = Path('/content/drive/MyDrive/qwen3_4b_numina_5k_smoketest/final_adapter')

required_adapter_files = ['adapter_config.json', 'adapter_model.safetensors']
missing = [name for name in required_adapter_files if not (DRIVE_ADAPTER_PATH / name).exists()]
if missing:
    raise FileNotFoundError(f'Missing adapter files under {DRIVE_ADAPTER_PATH}: {missing}')

run_cfg = Path('experiments/prompt_engineering/configs/run/sc_terse_only.yaml')
run_cfg.parent.mkdir(parents=True, exist_ok=True)
run_cfg.write_text('runs:\n  - prompt_id: sc_terse\n    regime: high_div_8\n')

print('Adapter:', DRIVE_ADAPTER_PATH)
print(run_cfg.read_text())

In [ ]:
import os
from pathlib import Path

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['RUNNER_PARALLEL_SAMPLES'] = '1'
os.environ['RUNNER_MICRO_BATCH_SIZE'] = '2'
os.environ['EVAL_METRICS_DIR'] = '/content/drive/MyDrive/qwen_eval_metrics'

# Uncomment after `wandb login`, or set these in Colab Secrets.
# os.environ['WANDB_PROJECT'] = 'qwen-sft-eval'
# os.environ['WANDB_ENTITY'] = 'YOUR_WANDB_ENTITY'
# os.environ['WANDB_GROUP'] = 'sc_terse_hf_bnb'

# Optional: persist Hugging Face downloads across Colab runtimes. Drive can be slower,
# so leave this commented unless re-downloading the model is the bigger annoyance.
# os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'

runner_text = Path('shared/runner.py').read_text()
if 'RUNNER_PARALLEL_SAMPLES' not in runner_text:
    raise RuntimeError('This checkout does not include the fast RUNNER_PARALLEL_SAMPLES runner patch.')

print('RUNNER_MICRO_BATCH_SIZE =', os.environ['RUNNER_MICRO_BATCH_SIZE'])
print('RUNNER_PARALLEL_SAMPLES =', os.environ['RUNNER_PARALLEL_SAMPLES'])

## Eval Helper

In [ ]:
import os
import shlex
import subprocess
import sys
import time

def run_eval(run_name, *, max_tokens, adapter=False, slice_indices=None, micro_batch_size=None):
    env = os.environ.copy()
    if adapter:
        env['ADAPTER_PATH'] = str(DRIVE_ADAPTER_PATH)
    else:
        env.pop('ADAPTER_PATH', None)
    if micro_batch_size is not None:
        env['RUNNER_MICRO_BATCH_SIZE'] = str(micro_batch_size)

    cmd = [
        sys.executable,
        '-m', 'experiments.prompt_engineering.src.eval',
        'run=sc_terse_only',
        f'eval.max_tokens={max_tokens}',
        'runner.engine=hf',
        'runner.quant=bnb',
        f'run_name={run_name}',
    ]
    if slice_indices is not None:
        joined = ','.join(str(i) for i in slice_indices)
        cmd.insert(4, f'eval.slice_indices=[{joined}]')

    print('ADAPTER_PATH =', env.get('ADAPTER_PATH', '<base model>'))
    print('RUNNER_MICRO_BATCH_SIZE =', env.get('RUNNER_MICRO_BATCH_SIZE'))
    print('RUNNER_PARALLEL_SAMPLES =', env.get('RUNNER_PARALLEL_SAMPLES'))
    print('Command:', ' '.join(shlex.quote(part) for part in cmd))
    start = time.time()
    subprocess.run(cmd, cwd=REPO_DIR, env=env, check=True)
    print(f'Elapsed: {(time.time() - start) / 60:.1f} min')

## 5-Item Speed Tests

A full 100-item run is about 20x the 5-item runtime. To stay under 4 hours, the 5-item speed test should finish in roughly 12 minutes or less.

In [ ]:
run_eval(
    'base_sc_terse_2048_5item_colab_speedtest',
    max_tokens=2048,
    adapter=False,
    slice_indices=[0, 1, 2, 3, 4],
)

In [ ]:
# Try this on A100 80GB if the 2048 run has plenty of time/headroom.
# run_eval(
#     'base_sc_terse_4096_5item_colab_speedtest',
#     max_tokens=4096,
#     adapter=False,
#     slice_indices=[0, 1, 2, 3, 4],
# )

## Full Base vs Adapter Runs

Set `MAX_TOKENS` based on the speed test. Use the same value for base and adapter.

In [ ]:
MAX_TOKENS = 2048  # Change to 4096 only if the 5-item speed test fits your budget.

run_eval(
    f'base_sc_terse_{MAX_TOKENS}_hf_colab_fast',
    max_tokens=MAX_TOKENS,
    adapter=False,
)

In [ ]:
run_eval(
    f'sft_numina5k_sc_terse_{MAX_TOKENS}_hf_colab_fast',
    max_tokens=MAX_TOKENS,
    adapter=True,
)

## Result Files

In [ ]:
from pathlib import Path

results_root = Path('experiments/prompt_engineering/results')
for path in sorted(results_root.glob('*'))[-10:]:
    print(path)